# 案例 3 · 滑动窗口神经网络预测（KUN 思想）

对应讲义 [第 11 讲](https://jiangyou2025.github.io/kun/course/11/)、[第 14 讲](https://jiangyou2025.github.io/kun/course/14/) 与 [使用 KUN](https://jiangyou2025.github.io/kun/zh/kun/)。

本案例演示深度预测最核心的骨架：
1. 把序列切成 (回看 L → 时域 H) 的训练样本；
2. 按训练集统计量归一化；
3. 训练一个**直接多步**线性预测器（DLinear 思想，也是 KUN 最简单的 kernel）；
4. 与季节性朴素基线对比。

> 这正是 KUN 的最小单元。KUN 在此之上，把序列切成 patch，用 U-Net 式的编码器/解码器把这种 kernel **层次化**地堆叠起来。

> 依赖：`numpy`, `matplotlib`, `torch`（`pip install torch`）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(0); torch.manual_seed(0)
n = 3000; t = np.arange(n)
# 类小时数据：日周期 24 + 周周期 168
y = (20 + 0.01*t + 5*np.sin(2*np.pi*t/24) + 8*np.sin(2*np.pi*t/168)
     + np.random.normal(0, 1.0, n)).astype('float32')
print('series length:', len(y))

## 1. 构造滑动窗口样本
回看一周 `L = 168`，预测未来一天 `H = 24`。

In [ ]:
L, H = 168, 24
def make_windows(series, L, H):
    X, Y = [], []
    for i in range(len(series) - L - H + 1):
        X.append(series[i:i+L])
        Y.append(series[i+L:i+L+H])
    return np.stack(X), np.stack(Y)

X, Y = make_windows(y, L, H)
print('X:', X.shape, ' Y:', Y.shape)

## 2. 按时间切分 + 归一化
用**训练集**的均值/标准差归一化，避免数据泄漏。

In [ ]:
ntr = int(len(X) * 0.7)
Xtr, Ytr = X[:ntr], Y[:ntr]
Xte, Yte = X[ntr:], Y[ntr:]

mu, sd = Xtr.mean(), Xtr.std()
norm = lambda v: (v - mu) / sd
Xtr_t = torch.tensor(norm(Xtr)); Ytr_t = torch.tensor(norm(Ytr))
Xte_t = torch.tensor(norm(Xte))
print('train windows:', len(Xtr), ' test windows:', len(Xte))

## 3. 一个最简单的 kernel：线性直接多步预测器

In [ ]:
class LinearForecaster(nn.Module):
    def __init__(self, L, H):
        super().__init__()
        self.fc = nn.Linear(L, H)
    def forward(self, x):
        return self.fc(x)

model = LinearForecaster(L, H)
model

## 4. 训练

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.L1Loss()                       # MAE，鲁棒且可解释
ds = torch.utils.data.TensorDataset(Xtr_t, Ytr_t)
dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True)

for epoch in range(20):
    model.train()
    for xb, yb in dl:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1:2d}  train L1 {loss.item():.4f}')

## 5. 评估：对比季节性朴素
因为 `H == 24` 且日周期为 24，季节性朴素 = 直接重复回看窗口最后 24 步。

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(Xte_t).numpy() * sd + mu          # 反归一化

mae_model  = np.mean(np.abs(pred - Yte))
snaive     = Xte[:, -24:]                           # 季节性朴素
mae_snaive = np.mean(np.abs(snaive - Yte))
print(f'MAE  KUN-style linear : {mae_model:.3f}')
print(f'MAE  seasonal-naive   : {mae_snaive:.3f}')

## 6. 可视化一个预测窗口

In [ ]:
i = 0
plt.figure(figsize=(11, 4))
plt.plot(range(L), Xte[i], label='lookback')
plt.plot(range(L, L+H), Yte[i], color='black', label='actual')
plt.plot(range(L, L+H), pred[i], '--', label='forecast')
plt.axvline(L, color='gray', ls=':')
plt.legend(); plt.title('One forecast window (L -> H)')
plt.tight_layout(); plt.show()

## 从这里到 KUN
- 这个线性层就是一个最简单的 **kernel**：把长度 L 的输入直接映射到长度 H 的输出。
- **Kernel U-Net (KUN)** 把序列切成 patch，用 U-Net 式的编码器逐层下采样、解码器逐层上采样，并在每个节点放一个可插拔 kernel（线性 / MLP / 注意力）。
- 想用真正的 KUN，请见 [使用 KUN](https://jiangyou2025.github.io/kun/zh/kun/)，把上面的 `LinearForecaster` 换成 `KernelUNet` 即可，训练与评估流程不变。